<a href="https://colab.research.google.com/github/subudear/deep-learning/blob/main/assignment2/partA_method2/birdnet_true_finetune_original_data_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
Part A Method 2 - True BirdNET fine-tuning on original data (v5 conservative fine-tune)
============================================================

This script replaces the earlier ImageNet EfficientNet transfer-learning model with
BirdNET+ V3.0 Developer Preview 3 Global 11K. The BirdNET model consumes raw audio
at 32 kHz and returns BirdNET embeddings plus predictions. We add a new assignment
classifier head and optionally unfreeze the BirdNET backbone for true fine-tuning.

V5 stability fixes:
- Uses stable validation loss accumulation and skips non-finite logits if they ever appear.
- Avoids Colab/Jupyter multiprocessing DataLoader cleanup errors by defaulting NUM_WORKERS to 0.
- Keeps the BirdNET backbone in eval mode during fine-tuning while still allowing gradients through its weights.
- Uses a much smaller backbone learning rate because full BirdNET unfreezing was reducing validation accuracy.
- Saves the best checkpoint by validation accuracy first, then validation loss as a tie-breaker.

Important reporting wording:
- Use: "BirdNET+ V3.0 transfer learning / fine-tuning on original data."
- Do not use: "ImageNet EfficientNet fine-tuning."

The EDA/reporting sections from the previous script are preserved and improved:
- Top 20 classes by sample count
- Sample-per-class distribution
- Audio property analysis
- Test preprocessing waveform + mel-spectrogram
- Data augmentation visualizations
- Training curves
- Confusion matrix and single-example class analysis

Colab usage:
1. Upload/open this .py as a notebook, or paste sections into Colab cells.
2. Update PROJECT_DIR, AUDIO_ROOT, METADATA_PATH, SPLIT_PATH if needed.
3. Use a GPU runtime.
"""

# %% [markdown]
# # 1. Mount Google Drive and set paths

import os
import sys
import math
import time
import json
import pickle
import random
import shutil
import subprocess
import urllib.request
from pathlib import Path
from typing import List, Tuple, Optional, Dict

# Mount Google Drive when running in Colab.
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
    print("✓ Google Drive mounted")
except Exception:
    IN_COLAB = False
    print("Not running in Colab or Drive already unavailable; continuing without drive.mount().")

# -----------------------------
# Update these paths if required
# -----------------------------
PROJECT_DIR = Path("/content/drive/MyDrive/audio_assignment/PartA_Method2_BirdNET_V3")
AUDIO_ROOT = Path("/content/drive/MyDrive/audio_assignment/train_audio")
METADATA_PATH = Path("/content/drive/MyDrive/audio_assignment/zip/train.csv")
SPLIT_PATH = Path("/content/drive/MyDrive/audio_assignment/PartA_Method2_BirdNET_V3/splits/train_validation_split.csv")

LABEL_COLUMN = "primary_label"
FILENAME_COLUMN = "filename"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "models").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "cache").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "results").mkdir(parents=True, exist_ok=True)

# Some archives extract as train_audio/train_audio. Resolve that automatically.
matches = list(AUDIO_ROOT.rglob("train_audio")) if AUDIO_ROOT.exists() else []
AUDIO_DIR = matches[0] if matches else AUDIO_ROOT

print(f"Project folder: {PROJECT_DIR}")
print(f"Audio folder:   {AUDIO_DIR}")
print(f"Metadata CSV:   {METADATA_PATH}")
print(f"Split CSV:      {SPLIT_PATH}")
print(f"Audio exists:   {AUDIO_DIR.exists()}")

# %% [markdown]
# # 2. Install/import dependencies

# Avoid reinstalling torch in Colab unless it is missing, because reinstalling torch can break CUDA.
def pip_install_if_missing(import_name: str, package_name: Optional[str] = None):
    package_name = package_name or import_name
    try:
        __import__(import_name)
    except ImportError:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

for import_name, package_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("sklearn", "scikit-learn"),
    ("tqdm", "tqdm"),
    ("librosa", "librosa>=0.10"),
    ("soundfile", "soundfile"),
    ("audioread", "audioread"),
]:
    pip_install_if_missing(import_name, package_name)

try:
    import torch
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch>=2.1"])
    import torch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import librosa
import librosa.display
import soundfile as sf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ GPU not available. True BirdNET fine-tuning will be slow on CPU.")

# %% [markdown]
# # 3. BirdNET+ V3.0 model download

# BirdNET+ V3.0 Developer Preview 3 Global 11K.
# The official birdnet-V3.0-dev analyze.py uses these default URLs.
BIRDNET_MODEL_NAME = "BirdNET+ V3.0 Preview 3 Global 11K FP32"
BIRDNET_SR = 32000
BIRDNET_CHUNK_SECONDS = 3.0
BIRDNET_CHUNK_SAMPLES = int(BIRDNET_SR * BIRDNET_CHUNK_SECONDS)

BIRDNET_MODEL_PATH = PROJECT_DIR / "models" / "BirdNET+_V3.0-preview3_Global_11K_FP32.pt"
BIRDNET_LABELS_PATH = PROJECT_DIR / "models" / "BirdNET+_V3.0-preview3_Global_11K_Labels.csv"
BIRDNET_MODEL_URL = "https://zenodo.org/records/18247420/files/BirdNET+_V3.0-preview3_Global_11K_FP32.pt?download=1"
BIRDNET_LABELS_URL = "https://zenodo.org/records/18247420/files/BirdNET+_V3.0-preview3_Global_11K_Labels.csv?download=1"


def download_if_missing(url: str, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 0:
        print(f"✓ Found existing file: {dst.name} ({dst.stat().st_size / 1e6:.1f} MB)")
        return
    print(f"Downloading {dst.name}...")
    tmp = dst.with_suffix(dst.suffix + ".tmp")
    try:
        urllib.request.urlretrieve(url, tmp)
        tmp.replace(dst)
        print(f"✓ Downloaded {dst.name} ({dst.stat().st_size / 1e6:.1f} MB)")
    except Exception as e:
        if tmp.exists():
            tmp.unlink()
        raise RuntimeError(
            f"Could not download {dst.name}. You can manually download it and place it at:\n{dst}\nURL:\n{url}\nOriginal error: {e}"
        )


download_if_missing(BIRDNET_MODEL_URL, BIRDNET_MODEL_PATH)
download_if_missing(BIRDNET_LABELS_URL, BIRDNET_LABELS_PATH)

# %% [markdown]
# # 4. Load metadata and EDA

if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Metadata CSV not found: {METADATA_PATH}")

raw_df = pd.read_csv(METADATA_PATH)
print(f"Dataset shape: {raw_df.shape}")
print("First rows:")
print(raw_df.head())
print("Columns:", raw_df.columns.tolist())

if LABEL_COLUMN not in raw_df.columns or FILENAME_COLUMN not in raw_df.columns:
    raise ValueError(
        f"Required columns not found. Need LABEL_COLUMN='{LABEL_COLUMN}' and FILENAME_COLUMN='{FILENAME_COLUMN}'. "
        f"Available: {raw_df.columns.tolist()}"
    )

class_counts = raw_df[LABEL_COLUMN].value_counts()
print(f"Number of unique classes/species: {len(class_counts)}")
print("Class count statistics:")
print(class_counts.describe())
print(f"Classes with 1 example: {(class_counts == 1).sum()}")
print(f"Classes with 2-5 examples: {((class_counts >= 2) & (class_counts <= 5)).sum()}")
print(f"Classes with >100 examples: {(class_counts > 100).sum()}")

# Top 20 and distribution plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
class_counts.head(20).sort_values().plot(kind="barh", ax=axes[0])
axes[0].set_title("Top 20 classes by sample count")
axes[0].set_xlabel("Number of samples")
axes[0].set_ylabel("Class")

axes[1].hist(class_counts.values, bins=50, edgecolor="black")
axes[1].set_title("Distribution of samples per class")
axes[1].set_xlabel("Samples per class")
axes[1].set_ylabel("Number of classes")
axes[1].set_yscale("log")
plt.tight_layout()
plt.savefig(PROJECT_DIR / "figures" / "class_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

print("\n⚠️ Class imbalance detected. The training code uses weighted cross-entropy and mixup.")

# %% [markdown]
# # 5. Audio property analysis for reporting


def resolve_audio_path(filename: str) -> Path:
    """Find an audio file robustly under AUDIO_DIR."""
    direct = AUDIO_DIR / filename
    if direct.exists():
        return direct
    # Some CSVs store relative paths; some archives have nested folders.
    rel = Path(filename)
    if rel.is_absolute() and rel.exists():
        return rel
    # Slow fallback; only used when direct lookup fails.
    matches = list(AUDIO_DIR.rglob(rel.name))
    return matches[0] if matches else direct


def analyze_audio_properties(df_in: pd.DataFrame, num_samples: int = 200) -> pd.DataFrame:
    results = []
    sample_df = df_in.sample(min(num_samples, len(df_in)), random_state=SEED)
    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Analyzing audio"):
        filepath = resolve_audio_path(str(row[FILENAME_COLUMN]))
        if not filepath.exists():
            continue
        try:
            y, sr = librosa.load(filepath, sr=None, mono=True)
            duration = librosa.get_duration(y=y, sr=sr)
            rms = librosa.feature.rms(y=y)[0]
            zcr = librosa.feature.zero_crossing_rate(y)[0]
            noise_duration = int(0.5 * sr)
            if len(y) > 2 * noise_duration:
                noise = np.concatenate([y[:noise_duration], y[-noise_duration:]])
                noise_power = float(np.mean(noise ** 2))
                signal_power = float(np.mean(y ** 2))
                snr_db = 10 * np.log10(signal_power / noise_power) if noise_power > 0 else np.inf
            else:
                snr_db = np.nan
            results.append({
                "filename": row[FILENAME_COLUMN],
                "label": row[LABEL_COLUMN],
                "sampling_rate": sr,
                "duration": duration,
                "num_samples": len(y),
                "min_rms": float(np.min(rms)),
                "mean_rms": float(np.mean(rms)),
                "mean_zcr": float(np.mean(zcr)),
                "snr_db": snr_db,
            })
        except Exception as e:
            print(f"Error processing {filepath}: {e}")
    return pd.DataFrame(results)


df_audio = analyze_audio_properties(raw_df, num_samples=200)
df_audio.to_csv(PROJECT_DIR / "results" / "audio_property_sample.csv", index=False)

if len(df_audio) > 0:
    print("=" * 70)
    print("AUDIO PROPERTIES ANALYSIS")
    print("=" * 70)
    print("Sampling rate distribution:")
    print(df_audio["sampling_rate"].value_counts())
    print("Duration statistics:")
    print(df_audio["duration"].describe())
    print("Background/noise features:")
    print(df_audio[["min_rms", "mean_rms", "mean_zcr", "snr_db"]].describe())

    fig, axes = plt.subplots(2, 3, figsize=(17, 10))
    axes[0, 0].hist(df_audio["sampling_rate"], bins=20, edgecolor="black")
    axes[0, 0].set_title("Sampling rate distribution")
    axes[0, 0].set_xlabel("Sampling rate (Hz)")
    axes[0, 0].set_ylabel("Count")

    axes[0, 1].hist(df_audio["duration"], bins=30, edgecolor="black")
    axes[0, 1].set_title("Audio duration distribution")
    axes[0, 1].set_xlabel("Duration (seconds)")
    axes[0, 1].set_ylabel("Count")

    axes[0, 2].hist(df_audio["min_rms"], bins=30, edgecolor="black")
    axes[0, 2].set_title("Background noise floor")
    axes[0, 2].set_xlabel("Minimum RMS")
    axes[0, 2].set_ylabel("Count")

    axes[1, 0].hist(df_audio["mean_rms"], bins=30, edgecolor="black")
    axes[1, 0].set_title("Mean RMS energy")
    axes[1, 0].set_xlabel("Mean RMS")
    axes[1, 0].set_ylabel("Count")

    axes[1, 1].hist(df_audio["mean_zcr"], bins=30, edgecolor="black")
    axes[1, 1].set_title("Zero crossing rate")
    axes[1, 1].set_xlabel("Mean ZCR")
    axes[1, 1].set_ylabel("Count")

    axes[1, 2].hist(df_audio["snr_db"].replace([np.inf, -np.inf], np.nan).dropna(), bins=30, edgecolor="black")
    axes[1, 2].set_title("Estimated signal-to-noise ratio")
    axes[1, 2].set_xlabel("SNR (dB)")
    axes[1, 2].set_ylabel("Count")
    plt.tight_layout()
    plt.savefig(PROJECT_DIR / "figures" / "audio_properties.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("⚠️ Audio property analysis found no readable files. Check AUDIO_DIR and filename column.")

# %% [markdown]
# # 6. Verify files and create/reuse train-validation split

# Resolve file paths.
df = raw_df.copy()
df["filepath"] = df[FILENAME_COLUMN].astype(str).apply(lambda x: str(resolve_audio_path(x)))
df["file_exists"] = df["filepath"].apply(lambda x: Path(x).exists())
print(f"Files found: {df['file_exists'].sum()} / {len(df)}")
df = df[df["file_exists"]].drop(columns=["file_exists"]).reset_index(drop=True)

if len(df) == 0:
    raise RuntimeError("No audio files found. Check AUDIO_ROOT/AUDIO_DIR and filename paths.")


def create_assignment_split(df_in: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Use existing split if present; otherwise create deterministic split."""
    if SPLIT_PATH.exists():
        split_df = pd.read_csv(SPLIT_PATH)
        possible_file_cols = [FILENAME_COLUMN, "filename", "filepath", "path", "file"]
        possible_split_cols = ["split", "fold", "set", "subset"]
        file_col = next((c for c in possible_file_cols if c in split_df.columns), None)
        split_col = next((c for c in possible_split_cols if c in split_df.columns), None)
        if file_col and split_col:
            print(f"✓ Reusing saved split: {SPLIT_PATH}")
            split_small = split_df[[file_col, split_col]].copy()
            split_small[file_col] = split_small[file_col].astype(str).apply(lambda p: Path(p).name)
            df_merge = df_in.copy()
            df_merge["_basename"] = df_merge[FILENAME_COLUMN].astype(str).apply(lambda p: Path(p).name)
            df_merge = df_merge.merge(split_small, left_on="_basename", right_on=file_col, how="left")
            df_merge[split_col] = df_merge[split_col].astype(str).str.lower()
            train_mask = df_merge[split_col].isin(["train", "training", "tr"])
            val_mask = df_merge[split_col].isin(["validation", "valid", "val", "dev"])
            train_out = df_merge[train_mask].drop(columns=["_basename", file_col, split_col], errors="ignore").reset_index(drop=True)
            val_out = df_merge[val_mask].drop(columns=["_basename", file_col, split_col], errors="ignore").reset_index(drop=True)
            if len(train_out) > 0 and len(val_out) > 0:
                return train_out, val_out
            print("⚠️ Existing split file was found but could not be matched cleanly. Creating a new split.")
        else:
            print("⚠️ Split file found, but filename/split columns were not recognized. Creating a new split.")

    counts = df_in[LABEL_COLUMN].value_counts()
    single_classes = counts[counts == 1].index.tolist()
    multi_classes = counts[counts > 1].index.tolist()
    df_single = df_in[df_in[LABEL_COLUMN].isin(single_classes)]
    df_multi = df_in[df_in[LABEL_COLUMN].isin(multi_classes)]

    train_multi, val_multi = train_test_split(
        df_multi,
        test_size=0.2,
        stratify=df_multi[LABEL_COLUMN],
        random_state=SEED,
    )
    train_out = train_multi.reset_index(drop=True)
    val_out = pd.concat([val_multi, df_single], axis=0).reset_index(drop=True)

    SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
    save_split = pd.concat([
        train_out[[FILENAME_COLUMN, LABEL_COLUMN]].assign(split="train"),
        val_out[[FILENAME_COLUMN, LABEL_COLUMN]].assign(split="validation"),
    ])
    save_split.to_csv(SPLIT_PATH, index=False)
    print(f"✓ Created and saved split: {SPLIT_PATH}")
    return train_out, val_out


train_df, val_df = create_assignment_split(df)

label_encoder = LabelEncoder()
label_encoder.fit(df[LABEL_COLUMN].astype(str))
train_df["label_encoded"] = label_encoder.transform(train_df[LABEL_COLUMN].astype(str))
val_df["label_encoded"] = label_encoder.transform(val_df[LABEL_COLUMN].astype(str))
num_classes = len(label_encoder.classes_)

print("=" * 70)
print("TRAIN/VALIDATION SPLIT")
print("=" * 70)
print(f"Training samples:   {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Classes total:      {num_classes}")
print(f"Classes in train:   {train_df[LABEL_COLUMN].nunique()}")
print(f"Classes in val:     {val_df[LABEL_COLUMN].nunique()}")

train_class_counts = train_df[LABEL_COLUMN].value_counts()
single_example_classes = df[LABEL_COLUMN].value_counts()[lambda s: s == 1].index.tolist()
print(f"Single-example classes placed in validation: {len(single_example_classes)}")

# Weighted CE for TRAINING only.
# Important: some classes are validation-only because they have one sample.
# Do not use zero-weight classes for validation loss, because weighted mean CE can become 0/0 => NaN.
weights = []
for cls in label_encoder.classes_:
    count = int(train_class_counts.get(cls, 0))
    if count <= 0:
        # Placeholder only. These labels do not appear in training.
        weights.append(1.0)
    else:
        # Inverse-sqrt weighting helps imbalance without exploding rare-class gradients.
        weights.append(1.0 / math.sqrt(count))
class_weights = torch.tensor(weights, dtype=torch.float32)
positive = class_weights[class_weights > 0]
if len(positive) > 0:
    class_weights = class_weights / positive.mean()
# Clip weights for stability; this is safer than very large rare-class weights.
class_weights = torch.clamp(class_weights, min=0.25, max=4.0)
print("✓ Stable class weights prepared for training loss")
print(f"  Weight range: {class_weights.min().item():.3f} to {class_weights.max().item():.3f}")

# %% [markdown]
# # 7. Preprocessing and augmentation visualizations for reporting


def sanitize_audio(y: np.ndarray) -> np.ndarray:
    """Make waveform safe for BirdNET and training. Keeps audio in a finite roughly [-1, 1] range."""
    y = np.asarray(y, dtype=np.float32)
    if y.size == 0:
        return y
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    max_abs = float(np.max(np.abs(y))) if y.size else 0.0
    if max_abs > 1.0:
        y = y / max_abs
    return np.clip(y, -1.0, 1.0).astype(np.float32)


def load_audio_resampled(filepath: str, sr: int = BIRDNET_SR) -> np.ndarray:
    y, _ = librosa.load(filepath, sr=sr, mono=True)
    return sanitize_audio(y)


def pad_or_crop(y: np.ndarray, target_len: int, mode: str = "random") -> np.ndarray:
    if len(y) == 0:
        return np.zeros(target_len, dtype=np.float32)
    if len(y) < target_len:
        return np.pad(y, (0, target_len - len(y)), mode="constant").astype(np.float32)
    if len(y) == target_len:
        return y.astype(np.float32)
    if mode == "center":
        start = (len(y) - target_len) // 2
    elif mode == "first":
        start = 0
    elif mode == "last":
        start = len(y) - target_len
    else:
        start = np.random.randint(0, len(y) - target_len + 1)
    return y[start:start + target_len].astype(np.float32)


def make_melspectrogram_for_plot(y: np.ndarray, sr: int = BIRDNET_SR):
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=512, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db


sample_row = train_df.iloc[0]
sample_path = sample_row["filepath"]
sample_label = sample_row[LABEL_COLUMN]
y_sample = load_audio_resampled(sample_path)
y_chunk = pad_or_crop(y_sample, BIRDNET_CHUNK_SAMPLES, mode="center")
mel_db = make_melspectrogram_for_plot(y_chunk)

print("Test preprocessing:")
print(f"  File: {Path(sample_path).name}")
print(f"  Label: {sample_label}")
print(f"  Raw duration: {len(y_sample) / BIRDNET_SR:.2f}s")
print(f"  BirdNET input chunk: {y_chunk.shape}, {BIRDNET_SR} Hz, {BIRDNET_CHUNK_SECONDS}s")
print(f"  Mel plot shape: {mel_db.shape}")

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
librosa.display.waveshow(y_chunk, sr=BIRDNET_SR, ax=axes[0])
axes[0].set_title(f"BirdNET input waveform: {sample_label}")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")
img = librosa.display.specshow(mel_db, sr=BIRDNET_SR, hop_length=512, x_axis="time", y_axis="mel", ax=axes[1])
axes[1].set_title("Mel-spectrogram view for reporting only; BirdNET consumes waveform directly")
fig.colorbar(img, ax=axes[1], format="%+2.0f dB")
plt.tight_layout()
plt.savefig(PROJECT_DIR / "figures" / "preprocessing_example.png", dpi=160, bbox_inches="tight")
plt.show()


class AudioAugmentation:
    @staticmethod
    def time_stretch(y: np.ndarray, rate: Optional[float] = None) -> np.ndarray:
        rate = float(rate if rate is not None else np.random.uniform(0.9, 1.1))
        return librosa.effects.time_stretch(y, rate=rate).astype(np.float32)

    @staticmethod
    def pitch_shift(y: np.ndarray, sr: int, n_steps: Optional[float] = None) -> np.ndarray:
        n_steps = float(n_steps if n_steps is not None else np.random.uniform(-2.0, 2.0))
        return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps).astype(np.float32)

    @staticmethod
    def add_noise(y: np.ndarray, noise_level: Optional[float] = None) -> np.ndarray:
        noise_level = float(noise_level if noise_level is not None else np.random.uniform(0.001, 0.004))
        return (y + np.random.randn(len(y)).astype(np.float32) * noise_level).astype(np.float32)

    @staticmethod
    def random_gain(y: np.ndarray, db_range: float = 6.0) -> np.ndarray:
        db = np.random.uniform(-db_range, db_range)
        return (y * (10 ** (db / 20))).astype(np.float32)

    @staticmethod
    def time_mask_for_plot(mel_spec: np.ndarray, max_width: int = 24) -> np.ndarray:
        out = mel_spec.copy()
        width = np.random.randint(1, min(max_width, out.shape[1]) + 1)
        start = np.random.randint(0, out.shape[1] - width + 1)
        out[:, start:start + width] = out.min()
        return out

    @staticmethod
    def freq_mask_for_plot(mel_spec: np.ndarray, max_height: int = 20) -> np.ndarray:
        out = mel_spec.copy()
        height = np.random.randint(1, min(max_height, out.shape[0]) + 1)
        start = np.random.randint(0, out.shape[0] - height + 1)
        out[start:start + height, :] = out.min()
        return out


aug = AudioAugmentation()
aug_examples = [
    ("Original", y_chunk),
    ("Time stretched", pad_or_crop(aug.time_stretch(y_chunk, rate=1.08), BIRDNET_CHUNK_SAMPLES, mode="center")),
    ("Pitch shifted", pad_or_crop(aug.pitch_shift(y_chunk, BIRDNET_SR, n_steps=2), BIRDNET_CHUNK_SAMPLES, mode="center")),
    ("Background noise", pad_or_crop(aug.add_noise(y_chunk, noise_level=0.003), BIRDNET_CHUNK_SAMPLES, mode="center")),
    ("Random gain", pad_or_crop(aug.random_gain(y_chunk), BIRDNET_CHUNK_SAMPLES, mode="center")),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.ravel()
for ax, (title, yy) in zip(axes[:5], aug_examples):
    m = make_melspectrogram_for_plot(yy)
    librosa.display.specshow(m, sr=BIRDNET_SR, hop_length=512, x_axis="time", y_axis="mel", ax=ax)
    ax.set_title(title)
# SpecAugment-style masks shown for report, but not fed to BirdNET waveform model.
masked = AudioAugmentation.freq_mask_for_plot(AudioAugmentation.time_mask_for_plot(mel_db))
librosa.display.specshow(masked, sr=BIRDNET_SR, hop_length=512, x_axis="time", y_axis="mel", ax=axes[5])
axes[5].set_title("SpecAugment view only")
plt.tight_layout()
plt.savefig(PROJECT_DIR / "figures" / "augmentation_examples.png", dpi=160, bbox_inches="tight")
plt.show()

# %% [markdown]
# # 8. Dataset and DataLoaders

TRAIN_AUGMENT_PROB = 0.35
BATCH_SIZE = 8 if torch.cuda.is_available() else 4
# Colab/Jupyter can throw multiprocessing cleanup warnings after long epochs.
# Use 0 for maximum stability. Increase to 2 later only if you need speed.
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()


class BirdNETWaveformDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, mode: str = "train", augment: bool = False):
        self.dataframe = dataframe.reset_index(drop=True)
        self.mode = mode
        self.augment = augment
        self.aug = AudioAugmentation()

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        y = load_audio_resampled(row["filepath"], sr=BIRDNET_SR)
        crop_mode = "random" if self.mode == "train" else "center"
        y = pad_or_crop(y, BIRDNET_CHUNK_SAMPLES, mode=crop_mode)

        if self.augment and np.random.rand() < TRAIN_AUGMENT_PROB:
            if np.random.rand() < 0.35:
                y = self.aug.time_stretch(y)
                y = pad_or_crop(y, BIRDNET_CHUNK_SAMPLES, mode="random")
            if np.random.rand() < 0.35:
                y = self.aug.pitch_shift(y, BIRDNET_SR)
            if np.random.rand() < 0.35:
                y = self.aug.add_noise(y)
            if np.random.rand() < 0.35:
                y = self.aug.random_gain(y)
            y = pad_or_crop(y, BIRDNET_CHUNK_SAMPLES, mode="random")

        y = sanitize_audio(y)

        # Keep waveform float32. BirdNET V3 expects [batch, samples] at 32 kHz.
        x = torch.tensor(y, dtype=torch.float32)
        y_label = torch.tensor(int(row["label_encoded"]), dtype=torch.long)
        return x, y_label


train_dataset = BirdNETWaveformDataset(train_df, mode="train", augment=True)
val_dataset = BirdNETWaveformDataset(val_df, mode="val", augment=False)

def make_loader(dataset, shuffle: bool):
    kwargs = dict(
        dataset=dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )
    # Only enable worker persistence when worker processes are actually used.
    if NUM_WORKERS > 0:
        kwargs.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(**kwargs)


train_loader = make_loader(train_dataset, shuffle=True)
val_loader = make_loader(val_dataset, shuffle=False)

print(f"Train dataset: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"Val dataset:   {len(val_dataset)} samples, {len(val_loader)} batches")

# %% [markdown]
# # 9. True BirdNET fine-tuning model

RUN_FULL_BIRDNET_FINETUNE = True
# Phase 2 is intentionally conservative. Your previous run showed that full unfreezing
# with a higher LR reduced validation accuracy and produced non-finite losses.

PHASE1_EPOCHS = 0      # Head only; set to 1 for a quick smoke test
PHASE2_EPOCHS = 3      # True BirdNET fine-tune, but conservative
EARLY_STOPPING_PATIENCE = 2
HEAD_LR = 2e-4
BACKBONE_LR = 2e-8     # Very small LR to avoid damaging pretrained BirdNET features
HEAD_LR_PHASE2 = 1e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 0.5
MIXUP_ALPHA = 0.0      # Keep disabled unless the stable baseline is already complete
USE_AMP = False        # Safer for TorchScript BirdNET fine-tuning; avoids mixed-precision NaNs

BEST_MODEL_PATH = PROJECT_DIR / "models" / "best_birdnet_v3_true_finetune.pt"
CHECKPOINT_PATH = PROJECT_DIR / "models" / "checkpoint_birdnet_v3_true_finetune.pt"


class BirdNETV3FineTuner(nn.Module):
    """BirdNET+ V3 backbone plus a new classifier head for this assignment."""

    def __init__(self, birdnet_model_path: Path, num_classes: int, device_for_init: torch.device):
        super().__init__()
        self.birdnet = torch.jit.load(str(birdnet_model_path), map_location=device_for_init)
        self.birdnet.eval()
        self._birdnet_trainable = False

        # Infer embedding dimension.
        with torch.no_grad():
            dummy = torch.zeros(1, BIRDNET_CHUNK_SAMPLES, dtype=torch.float32, device=device_for_init)
            out = self.birdnet(dummy)
            if not (isinstance(out, (tuple, list)) and len(out) >= 2):
                raise RuntimeError("Expected BirdNET V3 model to return (embeddings, predictions).")
            emb = out[0]
            if emb.ndim == 1:
                emb = emb.unsqueeze(0)
            embedding_dim = int(emb.shape[-1])

        self.embedding_dim = embedding_dim
        self.classifier = nn.Sequential(
            nn.LayerNorm(embedding_dim),
            nn.Dropout(0.35),
            nn.Linear(embedding_dim, 1024),
            nn.GELU(),
            nn.Dropout(0.30),
            nn.Linear(1024, 512),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(512, num_classes),
        )
        self.set_birdnet_trainable(False)

    def set_birdnet_trainable(self, trainable: bool):
        self._birdnet_trainable = bool(trainable)
        for p in self.birdnet.parameters():
            p.requires_grad = self._birdnet_trainable

        # Keep the pretrained BirdNET backbone in eval mode even during fine-tuning.
        # Gradients still flow when requires_grad=True, but dropout/batch-norm style
        # behaviour is fixed, which is much safer for small batch fine-tuning.
        self.birdnet.eval()

    def forward(self, x):
        if self._birdnet_trainable:
            out = self.birdnet(x)
        else:
            with torch.no_grad():
                out = self.birdnet(x)
        emb = out[0]
        if emb.ndim == 1:
            emb = emb.unsqueeze(0)
        logits = self.classifier(emb)
        return logits


model = BirdNETV3FineTuner(BIRDNET_MODEL_PATH, num_classes, device).to(device)
print(f"✓ Loaded {BIRDNET_MODEL_NAME}")
print(f"BirdNET embedding dimension: {model.embedding_dim}")
print(f"Assignment classes: {num_classes}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters initially: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
# Validation/reporting loss should be unweighted so validation-only classes do not create NaN.
criterion_eval = nn.CrossEntropyLoss(reduction="sum")


def mixup_batch(inputs, labels, alpha: float = MIXUP_ALPHA):
    if alpha <= 0:
        return inputs, labels, labels, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(inputs.size(0), device=inputs.device)
    mixed = lam * inputs + (1.0 - lam) * inputs[idx]
    return mixed, labels, labels[idx], float(lam)


def mixup_loss(criterion_fn, logits, y_a, y_b, lam):
    return lam * criterion_fn(logits, y_a) + (1.0 - lam) * criterion_fn(logits, y_b)


# %% [markdown]
# # 10. Training and validation helpers

if USE_AMP and device.type == "cuda":
    scaler = torch.amp.GradScaler("cuda", enabled=True)
else:
    scaler = None

from contextlib import nullcontext

def amp_context():
    if USE_AMP and device.type == "cuda":
        return torch.amp.autocast("cuda", enabled=True)
    return nullcontext()


def train_one_epoch(model, loader, optimizer, epoch_name: str, use_mixup: bool = True):
    model.train()
    # Keep BirdNET backbone in eval mode for stable fine-tuning, while classifier remains train().
    model.birdnet.eval()
    running_loss = 0.0
    total = 0
    correct = 0

    pbar = tqdm(loader, desc=epoch_name)
    for inputs, labels in pbar:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        do_mixup = use_mixup and inputs.size(0) > 1 and np.random.rand() < 0.5
        if do_mixup:
            inputs_m, y_a, y_b, lam = mixup_batch(inputs, labels)
        else:
            inputs_m, y_a, y_b, lam = inputs, labels, labels, 1.0

        with amp_context():
            logits = model(inputs_m)
            if not torch.isfinite(logits).all():
                print("⚠️ Non-finite training logits detected; skipping this batch.")
                continue
            loss = mixup_loss(criterion, logits, y_a, y_b, lam) if do_mixup else criterion(logits, labels)

        if not torch.isfinite(loss):
            print("⚠️ Non-finite training loss detected; skipping this batch.")
            continue

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()

        running_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()
        pbar.set_postfix(loss=f"{float(loss.detach().cpu()):.4f}", acc=f"{100.0 * correct / max(total, 1):.2f}%")

    return running_loss / max(total, 1), 100.0 * correct / max(total, 1)


@torch.no_grad()
def validate_one_epoch(model, loader, epoch_name: str = "Validation"):
    model.eval()
    running_loss = 0.0
    total = 0
    loss_total = 0
    correct = 0
    skipped_nonfinite = 0
    all_preds = []
    all_labels = []
    all_probs = []

    pbar = tqdm(loader, desc=epoch_name)
    for inputs, labels in pbar:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with amp_context():
            logits = model(inputs).float()

        finite_rows = torch.isfinite(logits).all(dim=1)
        if not finite_rows.all():
            skipped_nonfinite += int((~finite_rows).sum().item())
            logits = logits[finite_rows]
            labels = labels[finite_rows]
            if labels.numel() == 0:
                continue

        loss = criterion_eval(logits, labels)
        probs = torch.softmax(logits, dim=1)
        preds = logits.argmax(dim=1)

        if torch.isfinite(loss):
            running_loss += loss.item()
            loss_total += labels.size(0)
        else:
            print("⚠️ Non-finite validation loss detected; ignoring this batch loss.")

        total += labels.size(0)
        correct += (preds == labels).sum().item()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        pbar.set_postfix(loss=f"{float((loss / max(labels.size(0), 1)).detach().cpu()):.4f}", acc=f"{100.0 * correct / max(total, 1):.2f}%")

    if skipped_nonfinite > 0:
        print(f"⚠️ Skipped {skipped_nonfinite} validation sample(s) with non-finite logits.")

    return (
        running_loss / max(loss_total, 1),
        100.0 * correct / max(total, 1),
        np.asarray(all_preds),
        np.asarray(all_labels),
        np.asarray(all_probs),
    )


def save_checkpoint(path: Path, epoch: int, phase: str, val_loss: float, val_acc: float, history: Dict):
    torch.save({
        "epoch": epoch,
        "phase": phase,
        "model_state_dict": model.state_dict(),
        "val_loss": val_loss,
        "val_acc": val_acc,
        "label_classes": label_encoder.classes_.tolist(),
        "history": history,
        "birdnet_model_name": BIRDNET_MODEL_NAME,
        "birdnet_model_path": str(BIRDNET_MODEL_PATH),
        "sample_rate": BIRDNET_SR,
        "chunk_seconds": BIRDNET_CHUNK_SECONDS,
    }, path)


# %% [markdown]
# # 11. Phase 1: train the new classifier head only

history = {"epoch": [], "phase": [], "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
best_val_loss = float("inf")
best_val_acc = 0.0


def is_better_checkpoint(val_loss: float, val_acc: float) -> bool:
    if not np.isfinite(val_loss):
        return False
    return (val_acc > best_val_acc + 1e-6) or (abs(val_acc - best_val_acc) <= 1e-6 and val_loss < best_val_loss)

model.set_birdnet_trainable(False)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(PHASE1_EPOCHS, 1), eta_min=1e-5)

for epoch in range(1, PHASE1_EPOCHS + 1):
    print("=" * 80)
    print(f"PHASE 1 / Epoch {epoch}/{PHASE1_EPOCHS}: train classifier head only")
    print("=" * 80)
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, f"Phase1 Epoch {epoch}", use_mixup=(MIXUP_ALPHA > 0))
    val_loss, val_acc, val_preds, val_labels, val_probs = validate_one_epoch(model, val_loader)
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    history["epoch"].append(epoch)
    history["phase"].append("head_only")
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)

    print(f"Train loss {train_loss:.4f} | Train acc {train_acc:.2f}%")
    print(f"Val loss   {val_loss:.4f} | Val acc   {val_acc:.2f}% | LR {current_lr:.2e}")

    if is_better_checkpoint(val_loss, val_acc):
        best_val_loss = val_loss
        best_val_acc = val_acc
        save_checkpoint(BEST_MODEL_PATH, epoch, "head_only", val_loss, val_acc, history)
        print(f"✓ Saved best model: {BEST_MODEL_PATH}")

# %% [markdown]
# # 12. Phase 2: true BirdNET backbone fine-tuning

if RUN_FULL_BIRDNET_FINETUNE and PHASE2_EPOCHS > 0:
    if not BEST_MODEL_PATH.exists():
        print("⚠️ No best Phase 1 checkpoint exists yet; saving the current model as a fallback checkpoint.")
        save_checkpoint(BEST_MODEL_PATH, PHASE1_EPOCHS, "head_only_fallback", float("inf"), best_val_acc, history)
    checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    best_val_loss = float(checkpoint.get("val_loss", best_val_loss))
    best_val_acc = float(checkpoint.get("val_acc", best_val_acc))
    model.set_birdnet_trainable(True)
    print("✓ BirdNET backbone parameters are trainable. Backbone is kept in eval mode for stable true fine-tuning.")

    optimizer = torch.optim.AdamW([
        {"params": model.birdnet.parameters(), "lr": BACKBONE_LR},
        {"params": model.classifier.parameters(), "lr": HEAD_LR_PHASE2},
    ], weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(PHASE2_EPOCHS, 1), eta_min=1e-7)

    epochs_without_improvement = 0
    for epoch in range(1, PHASE2_EPOCHS + 1):
        global_epoch = PHASE1_EPOCHS + epoch
        print("=" * 80)
        print(f"PHASE 2 / Epoch {epoch}/{PHASE2_EPOCHS}: fine-tune BirdNET backbone + classifier")
        print("=" * 80)
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, f"Phase2 Epoch {epoch}", use_mixup=(MIXUP_ALPHA > 0))
        val_loss, val_acc, val_preds, val_labels, val_probs = validate_one_epoch(model, val_loader)
        scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]

        history["epoch"].append(global_epoch)
        history["phase"].append("birdnet_finetune")
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(current_lr)

        print(f"Train loss {train_loss:.4f} | Train acc {train_acc:.2f}%")
        print(f"Val loss   {val_loss:.4f} | Val acc   {val_acc:.2f}% | Backbone LR {current_lr:.2e}")

        save_checkpoint(CHECKPOINT_PATH, global_epoch, "birdnet_finetune", val_loss, val_acc, history)
        if is_better_checkpoint(val_loss, val_acc):
            best_val_loss = val_loss
            best_val_acc = val_acc
            epochs_without_improvement = 0
            save_checkpoint(BEST_MODEL_PATH, global_epoch, "birdnet_finetune", val_loss, val_acc, history)
            print(f"✓ Saved best model: {BEST_MODEL_PATH}")
        else:
            epochs_without_improvement += 1
            print(f"No improvement for {epochs_without_improvement} epoch(s)")

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break
else:
    print("⚠️ RUN_FULL_BIRDNET_FINETUNE is False. Completed head-only BirdNET transfer learning, not full backbone fine-tuning.")

print("=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Best validation accuracy: {best_val_acc:.2f}%")

# %% [markdown]
# # 13. Training history plot

history_df = pd.DataFrame(history)
history_df.to_csv(PROJECT_DIR / "results" / "training_history.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="s", label="Validation")
axes[0].axvline(PHASE1_EPOCHS + 0.5, linestyle="--", label="Fine-tune start")
axes[0].set_title("Training and validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_df["epoch"], history_df["train_acc"], marker="o", label="Train")
axes[1].plot(history_df["epoch"], history_df["val_acc"], marker="s", label="Validation")
axes[1].axvline(PHASE1_EPOCHS + 0.5, linestyle="--", label="Fine-tune start")
axes[1].set_title("Training and validation accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history_df["epoch"], history_df["lr"], marker="o")
axes[2].axvline(PHASE1_EPOCHS + 0.5, linestyle="--", label="Fine-tune start")
axes[2].set_title("Learning rate")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Learning rate")
axes[2].set_yscale("log")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_DIR / "figures" / "training_curves.png", dpi=160, bbox_inches="tight")
plt.show()

# %% [markdown]
# # 14. Standard validation evaluation

checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
# Keep model trainability as false for evaluation.
model.set_birdnet_trainable(False)
print(f"✓ Loaded best model from epoch {checkpoint['epoch']} ({checkpoint['phase']})")
print(f"  Val loss: {checkpoint['val_loss']:.4f}")
print(f"  Val acc:  {checkpoint['val_acc']:.2f}%")

val_loss, val_acc, all_preds, all_labels, all_probs = validate_one_epoch(model, val_loader, "Final validation")
val_accuracy = accuracy_score(all_labels, all_preds)
val_f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
val_f1_weighted = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

print("=" * 70)
print("STANDARD VALIDATION METRICS")
print("=" * 70)
print(f"Accuracy:          {val_accuracy * 100:.2f}%")
print(f"F1 macro:          {val_f1_macro:.4f}")
print(f"F1 weighted:       {val_f1_weighted:.4f}")
print("=" * 70)

report_dict = classification_report(
    all_labels,
    all_preds,
    labels=np.arange(num_classes),
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(PROJECT_DIR / "results" / "classification_report.csv")

print("Per-class performance for top 20 validation classes by sample count:")
val_class_counts = pd.Series(all_labels).value_counts()
for rank, (cls_idx, count) in enumerate(val_class_counts.head(20).items(), start=1):
    cls_name = label_encoder.classes_[cls_idx]
    mask = all_labels == cls_idx
    cls_acc = accuracy_score(all_labels[mask], all_preds[mask]) * 100
    print(f"{rank:2d}. {cls_name:30s} | samples={count:4d} | acc={cls_acc:6.2f}%")

# Confusion matrix for top classes by validation count.
top_n_classes = min(15, len(val_class_counts))
top_class_indices = val_class_counts.head(top_n_classes).index.tolist()
top_class_names = [label_encoder.classes_[i] for i in top_class_indices]
mask = np.isin(all_labels, top_class_indices)
filtered_labels = all_labels[mask]
filtered_preds = all_preds[mask]
# Keep only predictions that are also in the top classes for clearer matrix.
mask_pred = np.isin(filtered_preds, top_class_indices)
filtered_labels2 = filtered_labels[mask_pred]
filtered_preds2 = filtered_preds[mask_pred]

if len(filtered_labels2) > 0:
    cm = confusion_matrix(filtered_labels2, filtered_preds2, labels=top_class_indices)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=top_class_names, yticklabels=top_class_names)
    plt.title(f"Confusion matrix - top {top_n_classes} validation classes")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(PROJECT_DIR / "figures" / "confusion_matrix_top_classes.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("Could not create top-class confusion matrix because filtered predictions were empty.")

# Single-example class analysis.
single_indices = label_encoder.transform(single_example_classes) if single_example_classes else []
single_mask = np.isin(all_labels, single_indices)
if single_mask.sum() > 0:
    single_acc = accuracy_score(all_labels[single_mask], all_preds[single_mask]) * 100
    print("=" * 70)
    print("SINGLE-EXAMPLE CLASSES PERFORMANCE")
    print("=" * 70)
    print(f"Single-example validation samples: {single_mask.sum()}")
    print(f"Accuracy: {single_acc:.2f}%")
    print("These classes were not learnable from training data; report this limitation clearly.")
    print("=" * 70)

# %% [markdown]
# # 15. Multi-crop test-time augmentation validation

@torch.no_grad()
def predict_multicrop_for_row(row: pd.Series, n_crops: int = 5) -> np.ndarray:
    y = load_audio_resampled(row["filepath"], sr=BIRDNET_SR)
    if len(y) <= BIRDNET_CHUNK_SAMPLES:
        crops = [pad_or_crop(y, BIRDNET_CHUNK_SAMPLES, mode="center")]
    else:
        max_start = len(y) - BIRDNET_CHUNK_SAMPLES
        starts = np.linspace(0, max_start, n_crops).astype(int)
        crops = [y[s:s + BIRDNET_CHUNK_SAMPLES].astype(np.float32) for s in starts]
    x = torch.tensor(np.stack(crops), dtype=torch.float32, device=device)
    with amp_context():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).mean(dim=0)
    return probs.cpu().numpy()

RUN_TTA = True
TTA_CROPS = 5
if RUN_TTA:
    tta_probs = []
    tta_labels = []
    for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc=f"TTA validation ({TTA_CROPS} crops)"):
        tta_probs.append(predict_multicrop_for_row(row, n_crops=TTA_CROPS))
        tta_labels.append(int(row["label_encoded"]))
    tta_probs = np.asarray(tta_probs)
    tta_labels = np.asarray(tta_labels)
    tta_preds = np.argmax(tta_probs, axis=1)
    tta_accuracy = accuracy_score(tta_labels, tta_preds)
    tta_f1_macro = f1_score(tta_labels, tta_preds, average="macro", zero_division=0)
    tta_f1_weighted = f1_score(tta_labels, tta_preds, average="weighted", zero_division=0)
    print("=" * 70)
    print("FINAL VALIDATION METRICS WITH MULTI-CROP TTA")
    print("=" * 70)
    print(f"Accuracy:    {tta_accuracy * 100:.2f}%")
    print(f"F1 macro:    {tta_f1_macro:.4f}")
    print(f"F1 weighted: {tta_f1_weighted:.4f}")
    print("=" * 70)
else:
    tta_accuracy = np.nan
    tta_f1_macro = np.nan
    tta_f1_weighted = np.nan

# %% [markdown]
# # 16. Save final artifacts

metrics = {
    "model_type": "BirdNET+ V3.0 Developer Preview 3 Global 11K fine-tuning",
    "method": "A Method 2 - original data + true BirdNET fine-tuning",
    "num_classes": int(num_classes),
    "train_samples": int(len(train_df)),
    "validation_samples": int(len(val_df)),
    "best_val_loss": float(best_val_loss),
    "best_val_acc_percent": float(best_val_acc),
    "standard_accuracy_percent": float(val_accuracy * 100),
    "standard_f1_macro": float(val_f1_macro),
    "standard_f1_weighted": float(val_f1_weighted),
    "tta_accuracy_percent": None if np.isnan(tta_accuracy) else float(tta_accuracy * 100),
    "tta_f1_macro": None if np.isnan(tta_f1_macro) else float(tta_f1_macro),
    "tta_f1_weighted": None if np.isnan(tta_f1_weighted) else float(tta_f1_weighted),
    "birdnet_model_name": BIRDNET_MODEL_NAME,
    "birdnet_sample_rate": BIRDNET_SR,
    "birdnet_chunk_seconds": BIRDNET_CHUNK_SECONDS,
    "full_backbone_finetune": bool(RUN_FULL_BIRDNET_FINETUNE),
}

with open(PROJECT_DIR / "results" / "final_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

with open(PROJECT_DIR / "results" / "label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

pd.Series(label_encoder.classes_, name="class_name").to_csv(PROJECT_DIR / "results" / "class_names.csv", index=False)
train_df.to_csv(PROJECT_DIR / "results" / "train_df_used.csv", index=False)
val_df.to_csv(PROJECT_DIR / "results" / "val_df_used.csv", index=False)

print("✓ Saved artifacts:")
print(f"  Best model:             {BEST_MODEL_PATH}")
print(f"  Metrics JSON:           {PROJECT_DIR / 'results' / 'final_metrics.json'}")
print(f"  Training history CSV:   {PROJECT_DIR / 'results' / 'training_history.csv'}")
print(f"  Classification report:  {PROJECT_DIR / 'results' / 'classification_report.csv'}")
print(f"  Figures folder:         {PROJECT_DIR / 'figures'}")
print("\n🎉 Completed A Method 2 using true BirdNET model fine-tuning.")


Mounted at /content/drive
✓ Google Drive mounted
